# CS336 Assignment 1

## 实现BPE

### UNICODE

在UNICODE标准中，每个字符对应一个代码

In [3]:
ord("牛")
chr(29275)

'牛'

chr(0) 代表什么

In [ ]:
chr(0)
print(chr(0))

 


In [6]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [7]:
print("this is a test" + chr(0) + "string")

this is a test string


UNICODE标准中的0 代表空字符(NUL)，在python中， print会输出它，但通常终端不会显示出来
- 在C语言中用该字符作为字符串的结束标记

#### UNICODE 编码

将UNICODE字符用字节序列来进行编码，以解决词汇表太大的问题
- 直接使用UNICODE标准的代码，这个数字范围很大，几个UNICODE字符对应多少个代码

In [2]:
test_string0 = "A"
utf8_encoded = test_string0.encode("utf-16")
print(utf8_encoded)

test_string1 = "你好"
utf8_encoded = test_string1.encode("utf-8")
print(utf8_encoded)

b'\xff\xfeA\x00'
b'\xe4\xbd\xa0\xe5\xa5\xbd'


在解码时，UTF-8按照首字节的前缀，决定每个字符长度
- 0: 表示单字节字符，对应ASCII 字符
- 110: 后面跟一个续字节
- 1110: 后面跟两个
- 11110: 后面跟三个
- 续字节统一是10

因此一个UNICODE字符，并不对应一个字节
- 只有ASCII 字符是一个字符对应一个字节

而且这样所有的UNICODE字符我们都能用 0-255 来表示了，只要它们遵循统一的编码方式，如UTF-8
- 词汇表大小只有256

为什么 tokenizer 用UTF-8编码而不是UTF-16或UTF-32
- 因为ASCII在UTF-8 都是1字节，对英文场景天然更节省长度
- UTF-16 的基础单位是16位，但实际上一个可见字符可能占1个或2个单位

In [13]:
test_string= "hello 世界"
print(test_string.encode("utf-8"))
print(len(test_string.encode("utf-8")))
print(test_string.encode("utf-16"))
print(len(test_string.encode("utf-16")))


b'hello \xe4\xb8\x96\xe7\x95\x8c'
12
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00 \x00\x16NLu'
18


In [3]:
# A wrong func

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8")) ## no wrong
# decode_utf8_bytes_to_str_wrong("你好".encode("utf-8")) ## wrong


'hello'

这个函数错误的原因也就是前面提到的UTF-8编码的原理，除了ASCII，其他字符都不是一字节的

### 基于subword 的tokenization

由于基于字节的tokenization，会导致序列长度过长，也不是很适合

一个自然的思路就是，基于词出现的频率，将经常出现的，合并起来
- BPE

### BPE Tokenizer 训练

一般包括三步
1. 词表初始化: 词表是 token 到 整数的一一映射, 对于我们的字节级BPE分词器, 初始词表大小256, 对应一字节的所有可能取值
2. pre-tokenization

In [3]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

import regex as re

re.findall(PAT, "low lower")

['low', ' lower']

train_bpe
1. 读文件
   1. 输入：文件路径
   2. 输出：文本对象
2. 剥掉special token
   1. 输入：整个文本
   2. 输出：按special token 切成文档边界的预料
3. 正则预分词
   1. 输入：文本
   2. 输出：list
4. 每个词转UTF-8 字节
   1. 输入：str
   2. 输出：utf-8 表示
5. 统计相邻字节对频率
   1. 输入：整份语料
   2. 输出：计数表
6. 合并循环
   1. 输入：计数表 + 词表
   2. 输出：查计数表 -> 找频率最高的pair，把这对计入合并历史，在语料里把所以出现这个相邻对的地方替换成新词条，词表+1 检查是否达到vocab_size
7. 组装输出
   1. 输入：合并历史
   2. 输出：合并循环记下的所有oair，按创建顺序。256个单字节 + 新词表 + special token

In [8]:
old_tuple = (b'a', b'a', b'c')

pair = (b'a', b'a')

i = 0

new_tuple = []
while i < len(old_tuple):
    if i + 1 < len(old_tuple) and (old_tuple[i], old_tuple[i + 1]) == pair:
        new_tuple.append(old_tuple[i] + old_tuple[i+1])
        i += 2
    else:
        new_tuple.append(old_tuple[i])
        i += 1

new_tuple = tuple(new_tuple)
print(new_tuple)

(b'aa', b'c')


```
start = time.perf_counter()
vocab, merges = train_bpe('data/TinyStoriesV2-GPT4-train.txt', 10000, ['<|endoftext|>'])
end = time.perf_counter()
```

```
❯ uv run cs336_basics/tokenizer.py
72.07682266703341
```

TinyStoriesV2-GPT4-valid
- 总token数: 5465883
- 压缩率: 4.12 bytes/token
- encode 耗时: 10.01125 s

TinyStoriesV2-GPT4-train
- 总token数: 541229347
- 压缩率: 4.12 bytes/token
- encode 耗时: 1028.8971 s



# BPE Tokenizer 完整知识总结（2026-08 完成）

从零实现了一个完整的字节级 BPE tokenizer：训练 → 序列化 → 编码/解码，全部测试通过。
本笔记是"事后总结版"，用于复习和回答书面题。


## 一、BPE 全流程总览

```
训练（一次性，最贵）:   原始文本 → train_bpe → vocab(词表) + merges(合并规则，按创建顺序)
序列化:                vocab → JSON；merges → 文本（每行 "tok1 tok2"）
编码（高频使用）:       文本 → 预分词 → 按 merges 合并 → 查 vocab 得 ID
解码:                  ID → 查 vocab 得 bytes → 拼接 → utf-8 解码
```

三个关键认知：
1. **vocab 回答"token ↔ ID 的映射"；merges 回答"怎么把文本拆成 token"**——缺一不可
2. **encode = 重放训练**：训练时学到的 merge 规则，编码时按创建顺序应用
3. **merge 的顺序就是一切**：先并 (t,h) 再并 (th,e)，顺序反了结果就不同


## 二、训练算法（train_bpe）

### 数据流
```
读文件 → re.split 按 special token 切（硬边界，不跨文档合并）
→ PAT 正则预分词（GPT-2 模式，空格粘在词前面）
→ 每个 pre-token 转成单字节元组 (b'l', b'o', b'w')
→ 计数成 dict[tuple[bytes,...], int]（语料/工作台）
→ 合并循环 → 组装 vocab
```

### 合并循环（每轮 4 件事）
1. 查计数表 → 频率最高的 pair（**平局取字典序更大**，handout 的 max() 技巧）
2. 记入 merges（顺序 = 创建顺序）
3. 语料里把所有出现处合并（贪心从左到右，匹配到"吃掉两个"）
4. 词表 +1，直到 256 + merges数 + special_tokens数 == vocab_size

### 三个数据结构的分工
| 结构 | 存什么 | 回答什么问题 |
|---|---|---|
| corpus | pre-token → 频率 | 工作台本体 |
| pair_counts | pair → 加权总次数 | 合并谁（找 best）|
| pair_to_keys | pair → 含它的 pre-token 集合 | 去哪找（倒排索引）|

### 增量更新计数（性能关键）
- 朴素版每轮全量重算计数表 → O(轮数 × 语料)
- 优化：**只有贴着合并位置的 pair 会变**（3 个消失 + 2 个出现），其余不动
- 用"按 pre-token 差分"实现：旧元组 pair 贡献 -freq，新元组 +freq，归零就删
- 每个 pair **一生只被合并一次**（合并只组合元素、从不拆开）


## 三、优化旅程（实测数据，5M 样本 / vocab 1000）

| 版本 | 耗时 | 优化内容 |
|---|---|---|
| 朴素版 | 2.10 s | 每轮全量重算 pair 计数 |
| 增量差分 | 1.45 s | 只更新受影响的 pair |
| + 倒排索引 | 0.90 s | pair → pre-token 集合，不再全表扫描 |
| + 单字节缓存 | 再降 | bytes([b]) 改为查表 _SINGLE_BYTES |
| + 并行预分词 | 大文件生效 | multiprocessing + 按 <\|endoftext\|> 切块 |

**完整 TinyStories 训练（2.2GB / vocab 10000）：70 秒**（writeup 目标是 <2 分钟）

### 并行预分词的要点
- 用课程给的 find_chunk_boundaries 切块（边界落在 special token 开头，文档不被切断）
- worker 是模块顶层函数（macOS spawn 要求可 pickle）；父进程 Pool.map 分发后 get+freq 汇总
- **必须设阈值**：小文件走串行（spawn 启动 ~0.5s，corpus.en 并行反而更慢）
- 验证：强制并行 vs 串行，corpus dict 必须完全一致


## 四、Tokenizer 类（encode/decode）

### 接口（writeup §2.6.2）
```
__init__(vocab, merges, special_tokens=None)    # 建反向表 {bytes: id}
from_files(cls, vocab_path, merges_path, ...)   # 类方法 = 备用构造器（@classmethod）
encode(text) -> list[int]                       # 重放 merges
encode_iterable(iterable) -> Iterator[int]      # 流式，内存 O(1)
decode(ids) -> str                              # bytes 拼接 + utf-8 errors="replace"
```

### encode 的 rank 方法（tiktoken 的做法，性能关键）
- 朴素版：每个 pre-token 扫全部 merges → O(#merges × 词长)，2.2GB 跑几天
- **rank 方法**：`rank = {merge: i}`（越早 rank 越小）；循环内找"相邻 pair 中 rank 最小"的合并
- 正确性：新合并出的 pair 的 rank 一定 ≥ 当前（新元素 xy 刚诞生，含它的规则只能排在后面）
- 复杂度 O(词长²)，词长一般 <10 → 快两个数量级

### special token 处理
- **捕获组**：`re.split(f"({pattern})", text)` 把分隔符也保留，交替出现
- **按长度降序**排序再 join：正则交替"先到先得"，重叠 token（<|endoftext|><|endoftext|>）必须长的在前
- 遇到 special token 的 piece 直接查 ID，跳过预分词

### encode_iterable 的"接力空白"
- 朴素逐行编码 ≠ 整文编码：`\s+` 分支会让空白 run（如 \n\n）被行边界劈开
- 解法：`pending` 变量接力行尾空白 → rstrip 后编码非空白部分 → 尾部空白留给下一行
- 文件读完最后补编码 pending

### 内存对照组（测试的意义）
- encode 整个大文件 → 内存 O(n)（预期爆）
- encode_iterable → 内存 O(1)（1MB 上限内完成）
- Mac 上这俩测试跳过（RLIMIT_AS 在 macOS 不可靠），但**评分环境是 Linux 会跑**


## 五、序列化格式

- **为什么不能用原始 bytes 直接存**：JSON 不支持 bytes；且 merge 文本里 token 可能有空格
- **解法：GPT-2 字节→可见字符映射**（gpt2_bytes_to_unicode，课程提供）
  - 空格字节 → 'Ġ'，换行 → 'Ċ'，其余不可打印字节 → 各种可见符号
  - 这样 b' the' 存成 'Ġthe'，文本里没有真空格，按空格切分安全
- vocab → JSON（token字符串 → ID）；merges → 每行 "tok1 tok2"
- from_files = load_vocab_merges + cls(...)，**序列化让"昂贵的训练"变成"可复用资产"**
- round-trip 验证：save → load → assert 完全一致


## 六、实验结果（§2.7 数据）

| 项目 | 数值 |
|---|---|
| TinyStories 训练（2.2GB/10K 词表）| 70 秒 |
| 压缩率（train 和 valid）| 4.12 bytes/token |
| train 编码 token 数 | 541,229,347 |
| train 编码耗时 | ~17 分钟（吞吐 ≈ 2.2 MB/s）|
| 编码文件大小 | ≈ 1.08 GB（uint16，541M × 2 字节）|
| 最长 token | 15 字节：' accomplishment' / ' disappointment' / ' responsibility' |
| 词表大小 | 10000 |

**观察**：最长 token 是常见长单词（带头空格）——BPE 会把高频长词整词合并；大写开头的词（' Unfortunately'）是独立 token。


## 七、书面题素材速查

- **unicode1/2**：notes 前半部分（NUL 字符、UTF-8 前缀规则、decode_utf8_bytes_to_str_wrong 为何错）
- **train_bpe（15分）**：算法如上；special token 不参与统计但进词表（占 vocab_size 的坑）
- **train_bpe_tinystories (a)**：70s 训练；最长 token 15 字节；合理（常见长单词被整词合并）
- **train_bpe_tinystories (b)**：profile 显示预分词（genexpr/encode）和 max() 是大头；并行化预分词后显著下降
- **train_bpe_expts_owt**：16GB Mac 上 OOM（writeup 说需要 ≤100GB RAM）→ 低资源方案：采 500MB-1GB 子集训 32K
- **tokenizer_experiments**：压缩率 4.12；吞吐 2.2MB/s → Pile(825GB) 要 ~104 小时；uint16 因为 vocab 10000 < 65536


## 八、踩过的 Python 坑（血泪清单）

1. **`result.append[...]`**：方法调用用 `()`，`[]` 是下标
2. **`enc += 1`** 想改 dict 里的值却改了变量——`token_counts[enc] += 1`
3. **`d.get(k, set()).add(x)` 赋值**：`set.add()` 返回 None，赋回去就废了 → 用 `d.setdefault(k, set()).add(x)`
4. **`sorted()` 不改原列表**：要接住返回值（`x = sorted(...)`）；`list.sort()` 才是原地
5. **单元素元组必须有逗号**：`(x,)` 是元组，`(x)` 就是 x
6. **for 循环变量会泄漏**：循环结束 `tup` 还活着，容易误用成"上一个循环的残留值"
7. **找最小值要更新 min**：`if rank < min_rank:` 里忘了 `min_rank = rank` → 变成"最后一个"，不是"最小"
8. **`open()` 默认是只读**：写文件必须 `"w"`；`with` 保证关闭
9. **`f.write(...)` 没有换行**：所有行会挤在一起
10. **二进制模式不能带 encoding**：`open(p, "rb", encoding=...)` 报错 → 二进制读出后手动 `.decode()`
